# II2202 SNN-vs-ANN: main experiment (protocol P2) on Colab

Decision D-I (`docs/DECISIONS.md`): both SNNs are trained **and** evaluated at every T in {1, 2, 3, 6, 12, 24};
one ANN per seed (reused at every T); fine-tuning max 30 epochs; seeds {0, 1, 2}.

**Run one seed per Colab session** (about 10 h per seed on a T4, upper bound). Set `SEED` below.
The run is **resumable**: every finished (seed, T) is saved to Drive; after a disconnect, run all cells again
and it continues where it stopped. When all seeds are done, run the last cell (summary) once.

## 1. Settings

In [ ]:
SEED    = 0                                        # 0, 1 or 2: one per Colab session
RESULTS = '/content/drive/MyDrive/snn_results/main'  # shared by all seeds
DATA    = '/content/data'
REPO    = '/content/SNN-vs-ANN'
BRANCH  = 'feat/event-based-temporal-pipeline'

## 2. GPU and Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p {RESULTS}

## 3. Code (prints the exact commit) and packages

In [ ]:
import os
if os.path.isdir(REPO):
    !git -C {REPO} fetch -q origin && git -C {REPO} checkout -q {BRANCH} && git -C {REPO} pull -q --ff-only origin {BRANCH}
else:
    !git clone -q -b {BRANCH} https://github.com/liviacardaccia02/SNN-vs-ANN.git {REPO}
%cd {REPO}
!git log --oneline -1
!pip install -q tonic==1.7.0 spikingjelly==0.0.0.0.14 pytest

## 4. Fast tests (~4 min; optional on re-runs)

In [ ]:
!python -m pytest tests/test_opcount.py tests/test_models.py tests/test_conversion.py tests/test_training.py -q -p no:warnings

## 5. Dataset (~1.1 GB; skipped if present)

In [ ]:
if not (os.path.isdir(f'{DATA}/NMNIST/Train') and os.path.isdir(f'{DATA}/NMNIST/Test')):
    !python -c "import tonic; tonic.datasets.NMNIST('{DATA}', train=False); tonic.datasets.NMNIST('{DATA}', train=True)"
else:
    print('N-MNIST already on disk')

## 6. Main run for this seed (long; resumable)
Log is appended to Drive. If you see `CUDA out of memory`, add `--batch-size 64` and record it in `DECISIONS.md`.

In [ ]:
!python scripts/run_main.py --seeds {SEED} --data-root {DATA} --output-dir {RESULTS} 2>&1 | tee -a {RESULTS}/main_seed{SEED:02d}.log

## 7. Summary over all finished seeds (run at the end, or any time to see progress)

In [ ]:
!python scripts/run_main.py --summarize-only --output-dir {RESULTS}